# Parquet Basics — 09: Nested Data (Structs, Arrays, Maps)

## What you will learn
Real-world data is rarely flat. Parquet natively supports nested types
which Spark maps to StructType, ArrayType, and MapType.

In this notebook:
1. Writing and reading StructType (nested records)
2. Writing and reading ArrayType (lists of values)
3. Writing and reading MapType (key-value pairs)
4. Exploding nested data — `explode`, `posexplode`, `inline`
5. Accessing nested fields — dot notation and bracket notation
6. Schema inference for JSON → Parquet conversion


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:24:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


## Step 1 — StructType: Nested Records


In [2]:
from pyspark.sql.types import *

# Schema with nested struct
order_schema = StructType([
    StructField("order_id", LongType()),
    StructField("customer", StructType([
        StructField("id",      LongType()),
        StructField("name",    StringType()),
        StructField("address", StructType([
            StructField("city",    StringType()),
            StructField("country", StringType()),
            StructField("zip",     StringType()),
        ])),
        StructField("tier",    StringType()),
    ])),
    StructField("amount",    DoubleType()),
    StructField("status",    StringType()),
])

# Create data with nested structs
data = [
    (1, (101, "Alice Smith",  ("New York",     "US", "10001"), "gold"),   999.99,  "confirmed"),
    (2, (102, "Bob Jones",    ("London",        "UK", "EC1A"), "silver"),  499.99,  "shipped"),
    (3, (103, "Carol García", ("Berlin",        "DE", "10115"), "gold"),  1299.99, "delivered"),
    (4, (104, "Dave Lee",     ("Tokyo",         "JP", "100-0001"), "bronze"), 149.99, "pending"),
    (5, (105, "Eve Wang",     ("San Francisco", "US", "94105"), "gold"),   799.99,  "confirmed"),
]

df_nested = spark.createDataFrame(data, order_schema)

NESTED_PATH = f"{DATA_DIR}/nested"
df_nested.write.mode("overwrite").parquet(f"{NESTED_PATH}/structs")
print("Written nested struct Parquet")
df_nested.printSchema()
df_nested.show(truncate=False)

Written nested struct Parquet
root
 |-- order_id: long (nullable = true)
 |-- customer: struct (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- address: struct (nullable = true)
 |    |    |-- city: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- zip: string (nullable = true)
 |    |-- tier: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+--------+-------------------------------------------------+-------+---------+
|order_id|customer                                         |amount |status   |
+--------+-------------------------------------------------+-------+---------+
|1       |{101, Alice Smith, {New York, US, 10001}, gold}  |999.99 |confirmed|
|2       |{102, Bob Jones, {London, UK, EC1A}, silver}     |499.99 |shipped  |
|3       |{103, Carol García, {Berlin, DE, 10115}, gold}   |1299.99|delivered|
|4       |{104, Dave Lee, {Tokyo,

In [3]:
# Reading nested fields — dot notation
df_read = spark.read.parquet(f"{NESTED_PATH}/structs")

print("Access nested fields with dot notation:")
df_read.select(
    "order_id",
    "customer.name",
    "customer.tier",
    "customer.address.city",
    "customer.address.country",
    "amount"
).show(truncate=False)

# Column pruning applies to nested fields too!
print("\nColumn pruning — only reading customer.name and amount:")
df_read.select("order_id","customer.name","amount").explain(mode="formatted")

Access nested fields with dot notation:
+--------+------------+------+-------------+-------+-------+
|order_id|name        |tier  |city         |country|amount |
+--------+------------+------+-------------+-------+-------+
|1       |Alice Smith |gold  |New York     |US     |999.99 |
|3       |Carol García|gold  |Berlin       |DE     |1299.99|
|4       |Dave Lee    |bronze|Tokyo        |JP     |149.99 |
|5       |Eve Wang    |gold  |San Francisco|US     |799.99 |
|2       |Bob Jones   |silver|London       |UK     |499.99 |
+--------+------------+------+-------------+-------+-------+


Column pruning — only reading customer.name and amount:
== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [order_id#17L, customer#18, amount#19]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/parquet_basics/nested/structs]
ReadSchema: struct<order_id:bigint,customer:struct<name:string>,amount:double>

(2) ColumnarToRow [codegen

## Step 2 — ArrayType: Lists of Values


In [4]:
# Schema with array columns
product_schema = StructType([
    StructField("product_id", LongType()),
    StructField("name",       StringType()),
    StructField("tags",       ArrayType(StringType())),     # array of strings
    StructField("prices",     ArrayType(DoubleType())),     # array of doubles
    StructField("ratings",    ArrayType(StructType([        # array of structs!
        StructField("user_id", LongType()),
        StructField("score",   FloatType()),
    ]))),
])

product_data = [
    (1, "Laptop Pro",   ["tech","computing","premium"],  [999.99, 949.99, 1099.00],
     [(101, 4.5), (102, 5.0), (103, 4.0)]),
    (2, "Wireless Mouse",["tech","accessories","budget"],[29.99, 27.99],
     [(104, 3.5), (105, 4.5)]),
    (3, "Standing Desk", ["furniture","ergonomics"],     [399.99, 449.99, 379.00],
     [(101, 5.0), (103, 4.5), (106, 3.0)]),
]

df_products = spark.createDataFrame(product_data, product_schema)
df_products.write.mode("overwrite").parquet(f"{NESTED_PATH}/arrays")

print("Products with arrays:")
df_products.show(truncate=False)
df_products.printSchema()

Products with arrays:
+----------+--------------+---------------------------+------------------------+------------------------------------+
|product_id|name          |tags                       |prices                  |ratings                             |
+----------+--------------+---------------------------+------------------------+------------------------------------+
|1         |Laptop Pro    |[tech, computing, premium] |[999.99, 949.99, 1099.0]|[{101, 4.5}, {102, 5.0}, {103, 4.0}]|
|2         |Wireless Mouse|[tech, accessories, budget]|[29.99, 27.99]          |[{104, 3.5}, {105, 4.5}]            |
|3         |Standing Desk |[furniture, ergonomics]    |[399.99, 449.99, 379.0] |[{101, 5.0}, {103, 4.5}, {106, 3.0}]|
+----------+--------------+---------------------------+------------------------+------------------------------------+

root
 |-- product_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (contains

In [5]:
df_arr = spark.read.parquet(f"{NESTED_PATH}/arrays")

# Access array elements
print("Array element access:")
df_arr.select(
    "name",
    F.col("tags").getItem(0).alias("first_tag"),
    F.col("prices").getItem(0).alias("first_price"),
    F.size("tags").alias("tag_count"),
    F.array_contains("tags", "tech").alias("is_tech"),
).show(truncate=False)

# Explode: one row per array element
print("\nexplode(tags) — one row per tag:")
df_arr.select("name", F.explode("tags").alias("tag")).show()

# posexplode: includes position index
print("\nposexplode(prices) — includes position:")
df_arr.select("name", F.posexplode("prices").alias("pos","price")).show()

# Explode array of structs
print("\nexplode(ratings) — array of structs:")
df_arr.select("name",
    F.explode("ratings").alias("rating")
).select("name","rating.user_id","rating.score").show()

Array element access:
+--------------+---------+-----------+---------+-------+
|name          |first_tag|first_price|tag_count|is_tech|
+--------------+---------+-----------+---------+-------+
|Standing Desk |furniture|399.99     |2        |false  |
|Wireless Mouse|tech     |29.99      |3        |true   |
|Laptop Pro    |tech     |999.99     |3        |true   |
+--------------+---------+-----------+---------+-------+


explode(tags) — one row per tag:
+--------------+-----------+
|          name|        tag|
+--------------+-----------+
| Standing Desk|  furniture|
| Standing Desk| ergonomics|
|Wireless Mouse|       tech|
|Wireless Mouse|accessories|
|Wireless Mouse|     budget|
|    Laptop Pro|       tech|
|    Laptop Pro|  computing|
|    Laptop Pro|    premium|
+--------------+-----------+


posexplode(prices) — includes position:
+--------------+---+------+
|          name|pos| price|
+--------------+---+------+
| Standing Desk|  0|399.99|
| Standing Desk|  1|449.99|
| Standing Des

## Step 3 — MapType: Key-Value Pairs


In [6]:
# Schema with map column
event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("event_type", StringType()),
    StructField("properties", MapType(StringType(), StringType())),   # flexible metadata
    StructField("metrics",    MapType(StringType(), DoubleType())),   # numeric metrics
])

event_data = [
    (1, "purchase",
     {"product":"Laptop","currency":"USD","payment":"credit_card"},
     {"amount":999.99,"tax":89.99,"shipping":0.0}),
    (2, "page_view",
     {"page":"/products","device":"mobile","browser":"Chrome"},
     {"duration_sec":45.2,"scroll_pct":78.0}),
    (3, "search",
     {"query":"wireless headphones","results_count":"42","filters":"brand:Sony"},
     {"latency_ms":123.0,"click_through_rate":0.34}),
]

df_events = spark.createDataFrame(event_data, event_schema)
df_events.write.mode("overwrite").parquet(f"{NESTED_PATH}/maps")

print("Events with map columns:")
df_events.show(truncate=False)

Events with map columns:
+--------+----------+--------------------------------------------------------------------------+-------------------------------------------------+
|event_id|event_type|properties                                                                |metrics                                          |
+--------+----------+--------------------------------------------------------------------------+-------------------------------------------------+
|1       |purchase  |{product -> Laptop, payment -> credit_card, currency -> USD}              |{amount -> 999.99, tax -> 89.99, shipping -> 0.0}|
|2       |page_view |{page -> /products, device -> mobile, browser -> Chrome}                  |{duration_sec -> 45.2, scroll_pct -> 78.0}       |
|3       |search    |{filters -> brand:Sony, results_count -> 42, query -> wireless headphones}|{click_through_rate -> 0.34, latency_ms -> 123.0}|
+--------+----------+------------------------------------------------------------------------

In [7]:
df_maps = spark.read.parquet(f"{NESTED_PATH}/maps")

# Access map values
print("Map value access by key:")
df_maps.select(
    "event_id",
    "event_type",
    F.col("properties").getItem("product").alias("product"),
    F.col("metrics").getItem("amount").alias("amount"),
    F.map_keys("properties").alias("prop_keys"),
    F.map_values("metrics").alias("metric_values"),
).show(truncate=False)

# Explode map to key-value rows
print("\nexplode(properties) — one row per key-value pair:")
df_maps.select("event_id",
    F.explode("properties").alias("key","value")
).show()

Map value access by key:
+--------+----------+-------+------+-------------------------------+--------------------+
|event_id|event_type|product|amount|prop_keys                      |metric_values       |
+--------+----------+-------+------+-------------------------------+--------------------+
|3       |search    |NULL   |NULL  |[filters, results_count, query]|[0.34, 123.0]       |
|2       |page_view |NULL   |NULL  |[page, device, browser]        |[45.2, 78.0]        |
|1       |purchase  |Laptop |999.99|[product, payment, currency]   |[999.99, 89.99, 0.0]|
+--------+----------+-------+------+-------------------------------+--------------------+


explode(properties) — one row per key-value pair:
+--------+-------------+-------------------+
|event_id|          key|              value|
+--------+-------------+-------------------+
|       3|      filters|         brand:Sony|
|       3|results_count|                 42|
|       3|        query|wireless headphones|
|       2|         page

## Step 4 — JSON to Parquet: Nested Data in Practice

Real-world data often comes as JSON with deeply nested structures.
This is the production pattern for converting nested JSON to Parquet.


In [8]:
import json as pyjson, pathlib

# Simulate JSON events (like Kafka messages)
json_events = [
    {"event_id": i,
     "user": {"id": i % 1000, "country": ["US","UK","DE","FR"][i%4]},
     "items": [{"sku": f"SKU{j}", "price": round(j*10.5,2), "qty": j}
               for j in range(1, (i%4)+2)],
     "totals": {"subtotal": round(i*15.99,2), "tax": round(i*1.5,2)}
    } for i in range(1000)
]

json_path = f"{DATA_DIR}/events.json"
with open(json_path, "w") as f:
    for event in json_events:
        f.write(pyjson.dumps(event) + "\n")

# Let Spark infer the schema from JSON
df_json = spark.read.json(json_path)
print("Inferred schema from JSON:")
df_json.printSchema()

# Convert to Parquet (preserving nested structure)
pq_path = f"{DATA_DIR}/events_parquet"
df_json.write.mode("overwrite").parquet(pq_path)
print(f"\nConverted {df_json.count():,} events to Parquet")

# Query nested data from Parquet
df_pq = spark.read.parquet(pq_path)
print("\nTop countries by total subtotal:")
# After select(), nested dot-notation columns become flat names
# Use F.col() with dot-notation directly in groupBy/agg instead
df_pq.groupBy(F.col("user.country").alias("country")) \
     .agg(F.sum(F.col("totals.subtotal")).alias("total"),
          F.count("*").alias("events")) \
     .orderBy(F.desc("total")).show()

Inferred schema from JSON:
root
 |-- event_id: long (nullable = true)
 |-- items: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- price: double (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- sku: string (nullable = true)
 |-- totals: struct (nullable = true)
 |    |-- subtotal: double (nullable = true)
 |    |-- tax: double (nullable = true)
 |-- user: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- id: long (nullable = true)


Converted 1,000 events to Parquet

Top countries by total subtotal:
+-------+------------------+------+
|country|             total|events|
+-------+------------------+------+
|     FR|         2002747.5|   250|
|     DE|         1998750.0|   250|
|     UK|1994752.4999999998|   250|
|     US|         1990755.0|   250|
+-------+------------------+------+



## Summary: Nested Data Cheat Sheet

### Structs
```python
# Access nested field
df.select("customer.address.city")

# Flatten struct to columns
df.select("order_id", F.col("customer.*"))
```

### Arrays
```python
# Element access
F.col("tags").getItem(0)          # first element
F.size("tags")                    # array length
F.array_contains("tags", "tech")  # membership test

# Explode to rows
F.explode("tags")                 # one row per element
F.posexplode("tags")              # with position index
F.explode_outer("tags")           # keeps rows with empty/null arrays
```

### Maps
```python
# Key access
F.col("props").getItem("key")
F.map_keys("props")               # all keys as array
F.map_values("props")             # all values as array

# Explode to rows
F.explode("props")                # one row per key-value pair
```
